In [153]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [154]:
df=pd.read_csv(r"C:\Users\kirti\Desktop\AI Engineer\End-To-End Project\Movie Recommendation System\data\ratings.csv")

In [155]:
movies_meta=pd.read_csv(r"C:\Users\kirti\Desktop\AI Engineer\End-To-End Project\Movie Recommendation System\data\movies_metadata.csv")

C:\Users\kirti\AppData\Local\Temp\ipykernel_24424\1612413770.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_meta=pd.read_csv(r"C:\Users\kirti\Desktop\AI Engineer\End-To-End Project\Movie Recommendation System\data\movies_metadata.csv")


In [156]:
df.head()

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [157]:
movies_meta=movies_meta.rename(columns={"id":"movieId"})
movies_meta.head()

,adult,belongs_to_collection,budget,genres,homepage,movieId,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [158]:
movies_meta["movieId"]=pd.to_numeric(movies_meta["movieId"],errors="coerce") ## we use to_numeric() when we have to handle errors in data, .astype() throws error (only use when clean)

In [159]:
print(df.shape[0],movies_meta.shape[0])

26024289 45466


In [160]:
merg_df=pd.merge(df,movies_meta, on="movieId",how="inner") ## "inner" as i'll only need table that has both movie id and its name

In [161]:
import psutil
print(psutil.virtual_memory())

svmem(total=16785268736, available=6630932480, percent=60.5, used=10154336256, free=6630932480)


In [162]:
merg_df.columns

Index(['userId', 'movieId', 'rating', 'timestamp', 'adult',
       'belongs_to_collection', 'budget', 'genres', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'title', 'video', 'vote_average', 'vote_count'],
      dtype='object')

## Final Dataframe

In [163]:
fdf=merg_df[['userId', 'movieId', 'rating', 'timestamp', 'title']]
fdf

,userId,movieId,rating,timestamp,title
0,1,110,1.0,1425941529,Three Colors: Red
1,1,147,4.5,1425942435,The 400 Blows
2,1,858,5.0,1425941523,Sleepless in Seattle
3,1,1246,5.0,1425941556,Rocky Balboa
4,1,1968,4.0,1425942148,Fools Rush In
...,...,...,...,...,...
11437632,270896,48780,5.0,1257031830,Boat
11437633,270896,49530,4.0,1257034436,In Time
11437634,270896,54001,4.0,1257034331,The Traveler
11437635,270896,54503,4.0,1257033886,The Mystery of Chess Boxing


In [164]:
## fdf.pivot_table(index="userId",columns="movieId",values="rating")        Pivot table didn't worked because too many movies and 90 % + are sparse.

## Csr Matrix (as normal pivot table is huge and sparse)

In [165]:
from scipy.sparse import csr_matrix ## Compressed sparse rows
csr_matrix=csr_matrix((fdf["rating"],(fdf["userId"],fdf["movieId"])))
## Sparse matrix only stores non-zero values to save space and be memmory efficient.

#### Stored in this format:

- data = [3,2,1,4]
- indices = [4,2,3,6]
- indptr = [0,1,3,4]

#### Row 0 : 

data[0:1] = 3 rating

indices[0:1] = 4 th column

#### Row 1 : 

data[1:3] = 2,1 rating

indices[1:3] = 2nd, 3rd column

- User 0: [4, 0, 0]
- User 1: [0, 5, 0]
- User 2: [0, 0, 3]
- User 3: [0, 0, 0]  ← empty
- User 4: [0, 0, 0]  ← empty
- User 5: [2, 0, 0]
- User 6: [0, 1, 0]

In [166]:
print("Stored format:\ndata:",csr_matrix.data,"\n","indices:",csr_matrix.indices,"\n","indptr",csr_matrix.indptr)

Stored format:
data: [1.  4.5 5.  ... 4.  4.  5. ] 
 indices: [  110   147   858 ... 54001 54503 58559] 
 indptr [       0        0       11 ... 11436420 11436437 11436568]


In [167]:
train_matrix=csr_matrix.copy()

In [168]:
#train_matrix[test_rows,test_cols]=0
train_matrix.eliminate_zeros() ## removes zeros from storage

In [169]:
train_matrix.shape

(270897, 176274)

### Test-Set Split

In [170]:
test_size=round(0.2*train_matrix.nnz)
test_size

2287314

In [171]:
train_lil=train_matrix.tolil()
test_set=[]

for u in np.random.choice(range(train_matrix.shape[0]),train_lil.shape[0],replace=False):
    if len(test_set)>=test_size-1:
        break
    movies=train_lil.rows[u]
    data=train_lil.data[u]

    if len(movies)>3:
        idx=np.random.choice(len(movies),2,replace=False)
        for i in sorted(idx,reverse=True):
            test_set.append([u,movies[i],data[i]])
            del train_lil.rows[u][i]
            del train_lil.data[u][i]
            
            
train_matrix=train_lil.tocsr()

##### Lil is stored in this format :

A.rows = [ [0, 2], [0, 3], [1] ]

- User 0 rated item 0 and 2
- User 1 rated item 0 and 3
- User 2 rated item 1

A.data =
[ [5, 3], [4, 2], [1] ]

### Mean centering Matrix before SVD

In [172]:
user_mean=np.array(train_matrix.sum(axis=1)).flatten()/ train_matrix.getnnz(axis=1)
global_mean=train_matrix.data.mean()
user_mean[np.isnan(user_mean)]=global_mean
user_mean=user_mean.reshape(-1,1)
user_mean

C:\Users\kirti\AppData\Local\Temp\ipykernel_24424\324983227.py:1: RuntimeWarning: invalid value encountered in divide
  user_mean=np.array(train_matrix.sum(axis=1)).flatten()/ train_matrix.getnnz(axis=1)


array([[3.52965007],
       [4.11111111],
       [3.26666667],
       ...,
       [2.57142857],
       [4.2       ],
       [3.9496124 ]], shape=(270897, 1))

In [173]:
for i in range(train_matrix.shape[0]):
    start=train_matrix.indptr[i]
    end=train_matrix.indptr[i+1]
    train_matrix.data[start:end]-=user_mean[i,0]

How subtracting mean remove bias of users?

- SVD does decompostion of matrix into smaller matrix of latent feature which has least reconstruction error.
- Original matrix = latent matrix (user) x latent matrix (movie)
- Truncated SVD uses only top_n latent features to reconstruct matrix for better generalization and faster prediction.

### Model training

In [174]:
from sklearn.decomposition import TruncatedSVD ## Creates two matrix from single matrix having same latent features.
model=TruncatedSVD(n_components=50)
user_latent=model.fit_transform(train_matrix)  ## User latent features [U*∑]

Matrix Factorization idea shows up in:
- PCA
- SVD
- Word embeddings (Tf-idf Vectorizer)
- Transformers (attention factorization idea)

In [175]:
movie_latent=model.components_.T   ## Movie latent features (Transpose because reconstruction made movie latent transpose)

In [176]:
y_pred=[]
y_test=[]
for u,m,r in test_set:
    y_p=np.dot(user_latent[u],movie_latent[m]) +user_mean[u,0]
    y_t=r
    y_pred.append(y_p)
    y_test.append(y_t)

In [177]:
y_dumb = [global_mean] * len(y_test) 

why global mean?

In [182]:
from sklearn.metrics import mean_squared_error
dumb_rmse=np.sqrt(mean_squared_error(y_dumb,y_test))
rmse=np.sqrt(mean_squared_error(y_pred,y_test))
print("Dumb rmse:",dumb_rmse,"Real rmse:",rmse)

Dumb rmse: 1.1116527125831528 Real rmse: 1.0215529371935472


Why cosine be better sometimes rather than dot product?